# RSP Experiment Notebook — Cao et al. (2020) Reproduction

Run cells in order. Modify parameters in **§2 Configuration** to switch datasets or tune experiment settings.

This notebook is the preferred interactive entry point. It mirrors `run.py` and `src/experiment.py`, but keeps each stage inspectable.

## 1. Setup — Imports & Paths

In [ ]:
import gc
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from IPython.display import Markdown, display

# Ensure src/ is importable
PROJECT_ROOT = Path().resolve()  # or Path("./Cao_SOTA_MP").resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.graph import RoadNetwork
from src.generator import compute_deadline
from src.ilp_solver import solve_ilp
from src.milp_solver import solve_milp
from src.dijkstra_solver import solve_dijkstra
from src.experiment import (
    load_experiment_data,
    run_experiment,
    _compute_path_stats,
    _path_match,
)
from src.visualize import (
    plot_accuracy_vs_deadline,
    plot_probability_comparison,
    print_summary,
    save_compute_time_table,
)

try:
    import jinja2  # noqa: F401

    HAS_JINJA2 = True
except Exception:
    HAS_JINJA2 = False


def _format_percent_df(df: pd.DataFrame) -> pd.DataFrame:
    return df.apply(lambda col: col.map(lambda v: "" if pd.isna(v) else f"{v:.1%}"))


def show_table(
    df: pd.DataFrame, caption: str, fmt: str = "float", digits: int = 4
) -> None:
    display(Markdown(f"**{caption}**"))
    if HAS_JINJA2:
        if fmt == "percent":
            display(df.style.format("{:.1%}"))
        else:
            display(df.style.format(f"{{:.{digits}f}}"))
    else:
        if fmt == "percent":
            display(_format_percent_df(df))
        else:
            display(df.round(digits))


print(f"Project root: {PROJECT_ROOT}")
print(f"Imports OK. jinja2 available: {HAS_JINJA2}")


## 2. Configuration

Edit the dict below to switch datasets or tune parameters.

In [ ]:
# ============================================================
# Change these to match what you want to run
# ============================================================

# Option A: load pre-generated data (recommended)
DATA_DIR = "data/beijing_conflict"  # Beijing multi-level conflict (4 tiers × 2 variants)
# DATA_DIR = "data/full/seed524"  # conflict graph — dual edge types (fast_risky vs slow_stable)
# DATA_DIR = "data/full/seed42"   # artificial 65-node full experiment
# DATA_DIR = "data/beijing"      # Beijing OSM 587-node
# DATA_DIR = "data/small"        # 10-node debug

# Option B: inline generation (set to None, uses config below)
# DATA_DIR = None

# Solver settings
BACKEND = "SCIP"  # SCIP / CBC / GLPK
BIG_M = 1_000_000  # big-M cap (per-sample M bounded by this)
TIME_LIMIT = 60  # solver time limit (seconds)

# Experiment settings (used if DATA_DIR is None, or for overriding num_repeats)
ALPHAS = [0.5, 0.6, 0.7, 0.8, 0.9]
MAX_CANDIDATE_PATHS = 1000
DEADLINE_MODE = "heuristic"  # heuristic / exact
DEADLINE_ENUMERATION_CUTOFF = 15  # only used when DEADLINE_MODE='exact'

# Quick-run overrides: set to None to use all available data
MAX_REPEATS = 3     # None = all 10 repeats  (3×20×5=300 jobs within budget)
MAX_OD_PAIRS = 20   # None = all 20 OD pairs

# Output
OUTPUT_DIR = "results/beijing_conflict_scip/"
SAVE_CSV = True
GENERATE_PLOTS = True

# ============================================================

print(f"Data:      {DATA_DIR or '(inline generation)'}")
print(f"Backend:   {BACKEND}")
print(f"Alphas:    {ALPHAS}")
print(f"Deadline:  {DEADLINE_MODE}")
print(f"Output:    {OUTPUT_DIR}")
print(f"Overrides: MAX_REPEATS={MAX_REPEATS}, MAX_OD_PAIRS={MAX_OD_PAIRS}")
print(f"Expected:  {MAX_REPEATS} × {MAX_OD_PAIRS} × {len(ALPHAS)} = {MAX_REPEATS * MAX_OD_PAIRS * len(ALPHAS)} jobs")

## 3. Load (or Generate) Data

Loads `network.npz`, `travel_times.npz`, `od_pairs.npy`, `meta.yaml` from `DATA_DIR`.

In [ ]:
if DATA_DIR:
    data = load_experiment_data(DATA_DIR)
    network = data["network"]
    od_pairs_all = data["od_pairs"]
    travel_times_all = data["travel_times"]
    meta = data["meta"]
    print(f"Loaded: {network.num_nodes} nodes, {network.num_edges} edges")
    print(f"  OD pairs: {len(od_pairs_all)}")
    print(f"  Repeats:  {len(travel_times_all)}")
    print(f"  Samples:  {travel_times_all[0].shape[0]}")
    print(
        f"  Meta:     { {k: v for k, v in meta.items() if k not in ('travel_time_model',)} }"
    )

    # Full experiment defaults: run all repeats / all OD pairs in the dataset
    NUM_REPEATS = len(travel_times_all) if MAX_REPEATS is None else min(MAX_REPEATS, len(travel_times_all))
    NUM_OD = len(od_pairs_all) if MAX_OD_PAIRS is None else min(MAX_OD_PAIRS, len(od_pairs_all))
    od_pairs = od_pairs_all[:NUM_OD]
    travel_times = travel_times_all[:NUM_REPEATS]

    # Baseline seed (used in compute_deadline for per-sample SP sampling)
    BASE_SEED = meta.get("seed", 42)
else:
    # Inline generation (legacy, from config)
    from src.generator import (
        create_artificial_network,
        generate_travel_times,
        random_od_pairs,
    )

    network = create_artificial_network(65, 123, seed=42)
    od_pairs = random_od_pairs(network, 20, seed=42)
    travel_times = [
        generate_travel_times(network.num_edges, 500, (10.0, 100.0), seed=1000 + r)
        for r in range(3)
    ]
    NUM_REPEATS = len(travel_times)
    NUM_OD = len(od_pairs)
    BASE_SEED = 42

total_jobs = NUM_REPEATS * NUM_OD * len(ALPHAS)
print(
    f"\nWill run: {NUM_REPEATS} repeats × {NUM_OD} OD pairs × {len(ALPHAS)} α = {total_jobs} jobs"
)

### 3a. Inspect Network & Data

In [ ]:
# Quick look at the data
print(f"Nodes: {network.num_nodes}, Edges: {network.num_edges}")
print(f"OD pairs sample: {od_pairs[:3]}")
print(f"W shape (repeat 0): {travel_times[0].shape}")

# Edge stats for repeat 0
W0 = travel_times[0]
edge_means = W0.mean(axis=0)
edge_cvs = W0.std(axis=0) / edge_means
print(
    f"\nEdge travel time — mean: [{edge_means.min():.1f}, {edge_means.max():.1f}] min"
)
print(
    f"Edge CV               — median: {np.median(edge_cvs):.3f}, range: [{edge_cvs.min():.3f}, {edge_cvs.max():.3f}]"
)

## 4. Run a Single Job (Debug/Explore)

Tweak `REPEAT`, `OD_IDX`, `ALPHA` to run one specific case and inspect solver outputs.

In [ ]:
# ============================================================
# Change these to debug a specific case
REPEAT = 0  # which repeat
OD_IDX = 0  # which OD pair
ALPHA = 0.5  # deadline level
# ============================================================

W = travel_times[REPEAT]
N = W.shape[0]
o, d = od_pairs[OD_IDX]

print(f"Repeat={REPEAT}, OD=({o}→{d}), α={ALPHA}")
print(f"W shape: {W.shape}")
print()

# Compute deadline
t0 = time.perf_counter()
tau, tau_diag = compute_deadline(
    W,
    network,
    o,
    d,
    ALPHA,
    max_candidate_paths=MAX_CANDIDATE_PATHS,
    mode=DEADLINE_MODE,
    enumeration_cutoff=DEADLINE_ENUMERATION_CUTOFF,
    seed=BASE_SEED,
    return_diagnostics=True,
)
print(
    f"Deadline τ = {tau:.1f}  (T_min={tau_diag['T_min']:.1f}, T_max={tau_diag['T_max']:.1f}, candidates={tau_diag['candidate_count']}, mode={tau_diag['mode']})"
)

# Run ILP (ground-truth)
t0 = time.perf_counter()
ilp = solve_ilp(
    network, W, o, d, tau, big_m=BIG_M, solver_name=BACKEND, time_limit=TIME_LIMIT
)
t_ilp = time.perf_counter() - t0

# Run MILP
t0 = time.perf_counter()
milp = solve_milp(network, W, o, d, tau, solver_name=BACKEND, time_limit=TIME_LIMIT)
t_milp = time.perf_counter() - t0

# Run Dijkstra
t0 = time.perf_counter()
dij = solve_dijkstra(network, W, o, d, tau)
t_dij = time.perf_counter() - t0

# Report
print()
for name, r, t in [
    ("ILP", ilp, t_ilp),
    ("MILP", milp, t_milp),
    ("Dijkstra", dij, t_dij),
]:
    status = r["status"]
    prob = r["punctuality_prob"]
    late = r.get("lateness_count", "?")
    prob_str = f"{prob:.4f}" if prob is not None else "None"
    print(
        f"  {name:10s}: status={status:8s}  punct={prob_str}  late={late}/{N}  time={t:.3f}s"
    )

### 4a. Inspect Path Details

In [ ]:
if ilp["path_x"] is None:
    print("ILP did not return an optimal reference path for this case.")
else:
    # Which edges were chosen?
    ilp_edges = [
        (network.edges[j][0], network.edges[j][1])
        for j in np.where(ilp["path_x"] > 0.5)[0]
    ]
    dij_edges = [
        (network.edges[j][0], network.edges[j][1])
        for j in np.where(dij["path_x"] > 0.5)[0]
    ]

    print(f"ILP path ({len(ilp_edges)} edges): {ilp_edges}")
    print(f"Dij path ({len(dij_edges)} edges): {dij_edges}")
    print(f"Same path: {np.array_equal(ilp['path_x'], dij['path_x'])}")

    # Path travel time distributions
    ilp_times = W @ ilp["path_x"]
    dij_times = W @ dij["path_x"]
    print(
        f"\nILP path times — mean={ilp_times.mean():.1f}, max={ilp_times.max():.1f}, late={(ilp_times > tau).sum()}/{N}"
    )
    print(
        f"Dij path times — mean={dij_times.mean():.1f}, max={dij_times.max():.1f}, late={(dij_times > tau).sum()}/{N}"
    )

## 5. Run Full Experiment

Run all (repeat × OD × α) combinations. Progress printed per job.

In [ ]:
config = {
    "data": {"dir": DATA_DIR, "num_samples": travel_times[0].shape[0]},
    "experiment": {
        "num_repeats": NUM_REPEATS,
        "num_od_pairs": NUM_OD,
        "alphas": ALPHAS,
    },
    "solver": {
        "backend": BACKEND,
        "big_m": BIG_M,
        "time_limit": TIME_LIMIT,
    },
    "candidate_paths": {"max_paths": MAX_CANDIDATE_PATHS},
    "deadline": {
        "mode": DEADLINE_MODE,
        "enumeration_cutoff": DEADLINE_ENUMERATION_CUTOFF,
    },
}

df, aux = run_experiment(config, data_dir=DATA_DIR, return_aux=True)
deadline_diag_by_alpha = aux["deadline_diag_by_alpha"]
print(f"\nDone! {len(df)} rows")

## 6. Results Summary

In [ ]:
print_summary(df)

### 6a. Accuracy by α (Table)

In [ ]:
# Accuracy tables should only use rows with a valid ILP reference
acc_df = (
    df[df["reference_available"].fillna(False)]
    if "reference_available" in df.columns
    else df
)

# Tie-aware accuracy by alpha and method (primary metric)
tie_by_alpha = acc_df.pivot_table(
    index="alpha", columns="method", values="tie_aware_correct", aggfunc="mean"
)
show_table(
    tie_by_alpha, "Tie-Aware Accuracy by α (primary, |gap| ≤ 1/N)", fmt="percent"
)

# Path-match accuracy by alpha and method (auxiliary)
acc_by_alpha = acc_df.pivot_table(
    index="alpha", columns="method", values="correct", aggfunc="mean"
)
show_table(acc_by_alpha, "Path-Match Accuracy by α (auxiliary)", fmt="percent")

if "reference_available" in df.columns:
    ref_by_alpha = (
        df[df["method"] == "ILP"]
        .groupby("alpha")["reference_available"]
        .mean()
        .to_frame("ILP reference availability")
    )
    show_table(ref_by_alpha, "ILP Reference Availability by α", fmt="percent")

### 6b. Solve Time Statistics

In [ ]:
time_stats = df.groupby("method")["solve_time"].agg(["mean", "median", "max", "std"])
show_table(time_stats, "Solve Time Statistics (s)", fmt="float", digits=4)


### 6c. Objective Gap Distribution

In [ ]:
gap_data = df[df["method"] != "ILP"].dropna(subset=["objective_gap"])
print("Objective gap (p_ILP - p_method):")
for method in ["MILP", "Dijkstra"]:
    g = gap_data[gap_data["method"] == method]["objective_gap"]
    if len(g) > 0:
        print(
            f"  {method:10s}: mean={g.mean():.5f}  max={g.max():.5f}  min={g.min():.5f}"
        )

### 6d. Deadline Diagnostics

In [ ]:
print("=== Deadline Diagnostics ===")
for alpha in ALPHAS:
    diags = deadline_diag_by_alpha[alpha]
    ilp_rows = df[(df["method"] == "ILP") & (df["alpha"] == alpha)]
    ilp_puncts = ilp_rows["punctuality_prob"].dropna()
    all_cand_puncts = []
    for d in diags:
        all_cand_puncts.extend(d.get("candidate_puncts", []))
    cand_mean = np.mean(all_cand_puncts) if all_cand_puncts else float("nan")
    print(
        f"  α={alpha}: tau mean={np.mean([d['tau'] for d in diags]):.1f} "
        f"ILP_punct median={ilp_puncts.median():.3f} mean={ilp_puncts.mean():.3f} "
        f"min={ilp_puncts.min():.3f} max={ilp_puncts.max():.3f} | "
        f"cand_punct mean={cand_mean:.3f}"
    )

## 7. Plots

In [ ]:
if GENERATE_PLOTS:
    plot_accuracy_vs_deadline(df, f"{OUTPUT_DIR}/figures")
    plot_probability_comparison(df, f"{OUTPUT_DIR}/figures")

### 7a. Inline Plots (Optional)

Display plots directly in notebook.

In [ ]:
import matplotlib.pyplot as plt

# Primary: Tie-aware accuracy vs alpha
tie_acc = df.groupby(["alpha", "method"])["tie_aware_correct"].mean().reset_index()
tie_acc.columns = ["alpha", "method", "tie_accuracy"]

fig, ax = plt.subplots(figsize=(8, 5))
for method in ["ILP", "MILP", "Dijkstra"]:
    subset = tie_acc[tie_acc["method"] == method]
    ax.plot(subset["alpha"], subset["tie_accuracy"], "o-", label=method, markersize=8)
ax.set_xlabel("α (deadline level)")
ax.set_ylabel("Tie-Aware Accuracy (|gap| ≤ 1/N)")
ax.set_ylim(0, 1.1)
ax.legend()
ax.set_title("Tie-Aware Accuracy vs Deadline (primary metric)")
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Auxiliary: Path-match accuracy vs alpha
acc = df.groupby(["alpha", "method"])["correct"].mean().reset_index()
acc.columns = ["alpha", "method", "accuracy"]

fig, ax = plt.subplots(figsize=(8, 5))
for method in ["ILP", "MILP", "Dijkstra"]:
    subset = acc[acc["method"] == method]
    ax.plot(
        subset["alpha"],
        subset["accuracy"],
        "o--",
        label=method,
        markersize=6,
        alpha=0.7,
    )
ax.set_xlabel("α (deadline level)")
ax.set_ylabel("Path-Match Accuracy")
ax.set_ylim(0, 1.1)
ax.legend()
ax.set_title("Path-Match Accuracy vs Deadline (auxiliary)")
ax.grid(True, alpha=0.3)
plt.show()

## 8. Save Outputs

In [ ]:
from src.experiment import _save_worst_cases

if SAVE_CSV:
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    csv_path = f"{OUTPUT_DIR}/results.csv"
    df.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path}")
    save_compute_time_table(df, OUTPUT_DIR)
    _save_worst_cases(df, OUTPUT_DIR)

print("\nNotebook done.")

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "/home/kkk/projects/RSP/Cao_SOTA_MP/results/full_protocol_seed42_scip/results.csv"
)
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nMethods: {df['method'].unique()}")
print(f"Alphas: {sorted(df['alpha'].unique())}")
print(f"Repeats: {sorted(df['repeat'].unique())}")
print(f"OD pairs: {df['od_idx'].nunique()}")
print(f"Status values: {df['status'].value_counts().to_dict()}")
